<div style="border-left:6px solid #1a6040; padding:20px 28px; background:#f0f7f3; border-radius:0 12px 12px 0; margin-bottom:16px;">
<p style="color:#1a6040; font-size:12px; letter-spacing:3px; text-transform:uppercase; margin:0 0 8px 0; font-family:monospace; font-weight:600;">EXPLORATORY DATA ANALYSIS</p>
<h1 style="color:#0d2e1f; font-size:28px; font-weight:800; margin:0 0 10px 0; line-height:1.2;">Dengue Dynamics in Bangladesh</h1>
<p style="color:#3a5a48; font-size:14px; margin:0 0 20px 0; max-width:680px; line-height:1.7;">A longitudinal analysis of dengue fever across 8 divisions of Bangladesh (2014–2022), examining epidemic trends, seasonal transmission windows, regional disparities, and the correlation between climate variables and outbreak intensity.</p>
<span style="background:#d8ede2; color:#1a6040; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:600; margin-right:8px;">🗺️ 8 Divisions</span>
<span style="background:#d8ede2; color:#1a6040; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:600; margin-right:8px;">📅 2014 – 2022</span>
<span style="background:#d8ede2; color:#1a6040; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:600; margin-right:8px;">🦟 822 Records</span>
<span style="background:#d8ede2; color:#1a6040; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:600;">🐍 Python · Pandas · Seaborn</span>
</div>

## 📌 Background & Context

Dengue fever is a mosquito-borne viral infection transmitted primarily by *Aedes aegypti*. Bangladesh has experienced increasingly severe outbreaks over the past decade, with the **2019 epidemic** marking a turning point — over 100,000 reported cases in a single year, overwhelming the country's healthcare system.

Dengue transmission is tightly coupled to **climate**: warm temperatures accelerate the mosquito life cycle, while rainfall creates the standing-water breeding sites *Aedes* mosquitoes depend on. Understanding these dynamics is essential for early-warning systems and targeted vector control.

This analysis explores a merged dataset of monthly dengue case counts and climate readings across all 8 administrative divisions of Bangladesh.

> **Data sources:** Case data — Ministry of Health & Family Welfare (Bangladesh). Climate data — web-scraped divisional weather records. *Companion analysis to IEEE TENCON 2023 publication.*

## 📋 Table of Contents

1. [Setup & Configuration](#1)
2. [Data Loading & First Look](#2)
3. [Data Preparation](#3)
4. [Exploratory Data Analysis](#4)
   - 4.1 [Overview Statistics](#4-1)
   - 4.2 [Annual Epidemic Trend](#4-2)
   - 4.3 [Seasonal Transmission Window](#4-3)
   - 4.4 [Regional Distribution](#4-4)
   - 4.5 [Seasonal Heatmap (Month × Division)](#4-5)
   - 4.6 [Climate Correlation](#4-6)
   - 4.7 [Cases vs Deaths](#4-7)
   - 4.8 [Outlier Handling — Raw vs Resampled](#4-8)
5. [Key Findings](#5)

## 1. Setup & Configuration <a id='1'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Visual theme (green — public-health palette) ─────────────────────────────
GREEN      = '#1a6040'
GREEN_LT   = '#2e9e6a'
DARK       = '#1e293b'
ACCENT     = '#e8302a'

plt.rcParams.update({
    'figure.facecolor':  '#ffffff',
    'axes.facecolor':    '#f6faf8',
    'axes.edgecolor':    '#dce5e0',
    'axes.labelcolor':   '#2a4035',
    'axes.titlesize':    15,
    'axes.titleweight':  'bold',
    'axes.titlepad':     14,
    'axes.labelsize':    12,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'xtick.color':       '#5a7065',
    'ytick.color':       '#5a7065',
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'grid.color':        '#e6efe9',
    'grid.linewidth':    0.8,
    'legend.framealpha': 0.92,
    'legend.edgecolor':  '#dce5e0',
    'font.family':       'DejaVu Sans',
    'figure.dpi':        130,
})

MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

print("✅ Setup complete.")

## 2. Data Loading & First Look <a id='2'></a>

In [ ]:
df = pd.read_csv('dataset.csv')

print(f"Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns : {list(df.columns)}")
print()
df.head(8)

In [ ]:
print("=== Column Summary ===")
df.info()
print()
print("=== Missing values ===")
print(df.isnull().sum())

## 3. Data Preparation <a id='3'></a>

The dataset is already fairly clean. We parse dates, sort chronologically, add month/year helper columns, and confirm the divisional coverage.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

print("═" * 46)
print("  DATASET SUMMARY")
print("═" * 46)
print(f"  Records        : {len(df):,}")
print(f"  Divisions      : {df['Region'].nunique()}  ({', '.join(sorted(df['Region'].unique()))})")
print(f"  Date range     : {df['Date'].min().date()}  →  {df['Date'].max().date()}")
print(f"  Total cases    : {df['Total_Case'].sum():,.0f}")
print(f"  Total deaths   : {df['Total_Death'].sum():,.0f}")
print(f"  Case-fatality  : {df['Total_Death'].sum()/df['Total_Case'].sum()*100:.2f}%")
print("═" * 46)

## 4. Exploratory Data Analysis <a id='4'></a>

### 4.1 Overview Statistics <a id='4-1'></a>

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
fig.patch.set_facecolor('#ffffff')

worst_year = df.groupby('Year')['Total_Case'].sum().idxmax()
stats = [
    ('Total Cases',   f"{df['Total_Case'].sum()/1000:.0f}K", '🦟', GREEN),
    ('Total Deaths',  f"{df['Total_Death'].sum():.0f}",       '⚠️',  ACCENT),
    ('Peak Year',     f"{worst_year}",                         '📈', '#ff7e00'),
    ('Divisions',     f"{df['Region'].nunique()}",             '🗺️', GREEN_LT),
]
for ax, (label, value, icon, color) in zip(axes, stats):
    ax.set_facecolor(color + '12')
    for s in ax.spines.values():
        s.set_edgecolor(color + '44'); s.set_linewidth(1.5)
    ax.text(0.5, 0.66, value, transform=ax.transAxes, fontsize=28,
            fontweight='bold', ha='center', va='center', color=color)
    ax.text(0.5, 0.26, label, transform=ax.transAxes, fontsize=11,
            ha='center', va='center', color='#475569', fontweight='600')
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Dataset at a Glance', fontsize=15, fontweight='bold', color=DARK, y=1.05)
plt.tight_layout()
plt.savefig('dengue_overview.png', dpi=150, bbox_inches='tight')
plt.show()

df[['Avg Temperature','Avg Humidity','Avg Precipitation','Total_Case','Total_Death']].describe().round(1)

### 4.2 Annual Epidemic Trend <a id='4-2'></a>

Total reported dengue cases per year reveal the explosive, episodic nature of dengue in Bangladesh — long quiet periods punctuated by catastrophic epidemic years.

In [ ]:
yearly = df.groupby('Year')['Total_Case'].sum()

fig, ax = plt.subplots(figsize=(13, 5.5))
fig.patch.set_facecolor('#ffffff')

colors = [ACCENT if v == yearly.max() else GREEN for v in yearly.values]
bars = ax.bar(yearly.index.astype(str), yearly.values, color=colors,
              edgecolor='white', linewidth=1.5, width=0.68)

for bar, val in zip(bars, yearly.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + yearly.max()*0.015,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=10,
            fontweight='600', color='#2a4035')

# Annotate the 2019 epidemic
peak_year = yearly.idxmax()
ax.annotate('2019 epidemic - largest outbreak on record',
            xy=(list(yearly.index).index(peak_year), yearly.max()),
            xytext=(list(yearly.index).index(peak_year)-2.2, yearly.max()*0.82),
            fontsize=10, color=ACCENT, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.5))

ax.set_title('Reported Dengue Cases in Bangladesh by Year',
             fontsize=15, fontweight='bold', color=DARK, pad=16)
ax.text(0, 1.01, 'Dengue is episodic: the 2019 outbreak (101K cases) dwarfs all prior years, with 2021–22 marking a sustained resurgence.',
        transform=ax.transAxes, fontsize=10, color='#5a7065')
ax.set_ylabel('Total Reported Cases', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.5)
plt.tight_layout()
plt.savefig('dengue_annual.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Seasonal Transmission Window <a id='4-3'></a>

Averaging across all years reveals dengue's strong seasonal signature. Transmission peaks in the post-monsoon months when standing water and warm temperatures create ideal breeding conditions.

In [ ]:
monthly_avg = df.groupby('Month')['Total_Case'].mean()
monthly_std = df.groupby('Month')['Total_Case'].std()

# Clip lower error bar at 0 — case counts can't be negative
lower_err = np.minimum(monthly_std.values, monthly_avg.values)
upper_err = monthly_std.values
asym_err = [lower_err, upper_err]

fig, ax = plt.subplots(figsize=(13, 5.5))
fig.patch.set_facecolor('#ffffff')

peak_month = monthly_avg.idxmax()
colors = [ACCENT if m == peak_month else GREEN for m in monthly_avg.index]
bars = ax.bar(MONTH_NAMES, monthly_avg.values, yerr=asym_err,
              capsize=4, color=colors, edgecolor='white', linewidth=1.5,
              alpha=0.9, error_kw={'ecolor': '#9ab0a5', 'elinewidth': 1.2})

# Shade the monsoon season
ax.axvspan(4.5, 8.5, alpha=0.08, color='#3b82f6', zorder=0)
ax.text(6.5, ax.get_ylim()[1]*0.92, 'Monsoon (Jun-Sep)', ha='center',
        fontsize=9, color='#3b82f6', fontweight='600')

ax.set_ylim(0, None)
ax.set_title('Average Monthly Dengue Cases (2014-2022)',
             fontsize=15, fontweight='bold', color=DARK, pad=22)
ax.text(0, 1.02, 'Transmission surges from August, peaking Sep-Oct - the post-monsoon breeding window. Error bars show std across years.',
        transform=ax.transAxes, fontsize=10, color='#5a7065')
ax.set_ylabel('Mean Cases per Division', fontsize=12)
ax.grid(axis='y', alpha=0.5)
plt.tight_layout()
plt.savefig('dengue_seasonal.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Regional Distribution <a id='4-4'></a>

Dengue burden is highly unequal across divisions. Dhaka — the densely populated capital — bears the overwhelming majority of cases.

In [ ]:
region_cases = df.groupby('Region')['Total_Case'].sum().sort_values()

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#ffffff')

# Gradient from light to dark green by burden
norm = region_cases.values / region_cases.values.max()
colors = [plt.cm.YlOrRd(0.25 + 0.6*n) for n in norm]
bars = ax.barh(region_cases.index, region_cases.values, color=colors,
               edgecolor='white', linewidth=1.5, height=0.68)

total = region_cases.sum()
for bar, val in zip(bars, region_cases.values):
    pct = val/total*100
    ax.text(bar.get_width() + total*0.008, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}  ({pct:.1f}%)', va='center', fontsize=10,
            fontweight='600', color='#2a4035')

ax.set_title('Total Dengue Cases by Division (2014–2022)',
             fontsize=15, fontweight='bold', color=DARK, pad=16)
ax.text(0, 1.02, 'Dhaka accounts for ~72% of all cases nationwide — a hotspot driven by population density and urbanization.',
        transform=ax.transAxes, fontsize=10, color='#5a7065')
ax.set_xlabel('Total Reported Cases', fontsize=12)
ax.set_xlim(0, region_cases.max()*1.18)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='x', alpha=0.5)
plt.tight_layout()
plt.savefig('dengue_regional.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Seasonal Heatmap — Month × Division <a id='4-5'></a>

Combining the seasonal and regional views into a single heatmap shows *where* and *when* dengue concentrates.

In [ ]:
pivot = df.pivot_table(values='Total_Case', index='Month', columns='Region', aggfunc='mean')
pivot.index = MONTH_NAMES
# Order divisions by total burden
region_order = df.groupby('Region')['Total_Case'].sum().sort_values(ascending=False).index
pivot = pivot[region_order]

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#ffffff')

sns.heatmap(pivot, ax=ax, cmap='YlOrRd', linewidths=0.5, linecolor='white',
            annot=True, fmt='.0f', annot_kws={'size': 8.5, 'weight': 'bold'},
            cbar_kws={'label': 'Mean Cases', 'shrink': 0.82})

ax.set_title('Mean Monthly Dengue Cases by Division',
             fontsize=15, fontweight='bold', color=DARK, pad=16)
ax.text(0, 1.02, 'The Sep–Nov window in Dhaka is the epicenter of national dengue transmission.',
        transform=ax.transAxes, fontsize=10, color='#5a7065')
ax.set_xlabel(''); ax.set_ylabel('')
ax.tick_params(axis='x', rotation=25)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('dengue_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.6 Climate Correlation <a id='4-6'></a>

Do climate variables track dengue incidence? We examine pairwise correlations between temperature, humidity, precipitation, and case counts.

In [ ]:
corr_cols = ['Avg Temperature', 'Avg Humidity', 'Avg Precipitation', 'Total_Case', 'Total_Death']
labels = ['Avg Temp', 'Avg Humidity', 'Avg Precip', 'Total Cases', 'Total Deaths']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8.5, 7))
fig.patch.set_facecolor('#ffffff')

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, ax=ax, mask=mask, cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 11, 'weight': 'bold'},
            linewidths=0.6, linecolor='white', square=True,
            xticklabels=labels, yticklabels=labels,
            cbar_kws={'label': 'Correlation', 'shrink': 0.75})

ax.set_title('Climate–Dengue Correlation Matrix',
             fontsize=15, fontweight='bold', color=DARK, pad=16)
ax.text(0, 1.03, 'Humidity shows the strongest climate link to cases; deaths track cases almost perfectly (r≈0.95).',
        transform=ax.transAxes, fontsize=9.5, color='#5a7065')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('dengue_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.7 Cases vs Deaths Over Time <a id='4-7'></a>

Plotting the national monthly time series of cases against deaths shows how mortality tracks caseload — and highlights the human cost of each epidemic wave.

In [ ]:
monthly_ts = df.groupby('Date').agg(Cases=('Total_Case','sum'), Deaths=('Total_Death','sum'))

fig, ax1 = plt.subplots(figsize=(14, 5.5))
fig.patch.set_facecolor('#ffffff')

ax1.fill_between(monthly_ts.index, monthly_ts['Cases'], color=GREEN, alpha=0.18, zorder=1)
ax1.plot(monthly_ts.index, monthly_ts['Cases'], color=GREEN, linewidth=2, zorder=2, label='Cases')
ax1.set_ylabel('Monthly Cases', fontsize=12, color=GREEN)
ax1.tick_params(axis='y', labelcolor=GREEN)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2 = ax1.twinx()
ax2.plot(monthly_ts.index, monthly_ts['Deaths'], color=ACCENT, linewidth=1.8,
         zorder=3, label='Deaths', alpha=0.85)
ax2.set_ylabel('Monthly Deaths', fontsize=12, color=ACCENT)
ax2.tick_params(axis='y', labelcolor=ACCENT)
ax2.spines['top'].set_visible(False)
ax2.grid(False)

ax1.set_title('National Dengue Cases & Deaths Over Time',
              fontsize=15, fontweight='bold', color=DARK, pad=16)
ax1.text(0, 1.01, 'Each epidemic wave brings a corresponding mortality spike — deaths follow cases with tight coupling.',
         transform=ax1.transAxes, fontsize=10, color='#5a7065')
ax1.set_xlabel('Date', fontsize=12)
ax1.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('dengue_cases_deaths.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.8 Outlier Handling — Raw vs Resampled <a id='4-8'></a>

Epidemic years (2019, 2021, 2022) contain extreme case spikes that can dominate a model and obscure the underlying seasonal signal. The paper mitigates this by **resampling** — replacing outlier-year values with the baseline monthly average from non-epidemic years. This visual compares the raw and resampled national time series.

In [ ]:
OUTLIER_YEARS  = [2019, 2021, 2022]
BASELINE_YEARS = [2014, 2015, 2016, 2017, 2018, 2020]

# Baseline monthly average per region per month (from non-epidemic years)
baseline_df = df[df['Year'].isin(BASELINE_YEARS)]
baseline_avg = (baseline_df.groupby(['Region', 'Month'])['Total_Case']
                .mean().reset_index()
                .rename(columns={'Total_Case': 'Baseline_Avg'}))

# Build resampled dataset — replace outlier-year values with baseline
df_res = df.merge(baseline_avg, on=['Region', 'Month'], how='left')
df_res['Total_Case_Resampled'] = np.where(
    df_res['Year'].isin(OUTLIER_YEARS),
    df_res['Baseline_Avg'].fillna(df_res['Total_Case']),
    df_res['Total_Case']
)

raw_ts = df.groupby('Date')['Total_Case'].sum()
res_ts = df_res.groupby('Date')['Total_Case_Resampled'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
fig.patch.set_facecolor('#ffffff')

axes[0].fill_between(raw_ts.index, raw_ts.values, color=ACCENT, alpha=0.15)
axes[0].plot(raw_ts.index, raw_ts.values, color=ACCENT, linewidth=1.4)
axes[0].set_title('Before — Raw Data (with epidemic spikes)', fontsize=12, fontweight='bold', color=DARK)
axes[0].set_ylabel('Total Cases', fontsize=11)
axes[0].grid(alpha=0.4)

axes[1].fill_between(res_ts.index, res_ts.values, color=GREEN, alpha=0.15)
axes[1].plot(res_ts.index, res_ts.values, color=GREEN, linewidth=1.4)
axes[1].set_title('After — Resampled (outliers replaced)', fontsize=12, fontweight='bold', color=DARK)
axes[1].grid(alpha=0.4)

fig.suptitle('Outlier Mitigation via Baseline Resampling',
             fontsize=15, fontweight='bold', color=DARK, y=1.04)
fig.text(0.5, -0.02, 'Resampling flattens the extreme 2019/2021/2022 peaks while preserving the recurring seasonal structure.',
         ha='center', fontsize=10, color='#5a7065')
plt.tight_layout()
plt.savefig('dengue_resampled.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Key Findings <a id='5'></a>

---

### 🦟 1. Dengue is Episodic, Not Steady
Cases stayed under 11K/year through 2018, then exploded to **101K in 2019** — the largest outbreak on record. After a COVID-suppressed 2020, dengue resurged strongly in 2021–22, suggesting a troubling shift toward sustained high transmission.

---

### 🏙️ 2. Dhaka is the National Epicenter
The capital division accounts for roughly **72% of all reported cases** despite being one of 8 divisions. High population density, rapid urbanization, and abundant *Aedes* breeding sites concentrate the burden here.

---

### 🌧️ 3. Transmission Peaks Post-Monsoon
Cases surge from **August through October**, following the monsoon. Rainfall creates standing-water breeding sites; the lag reflects the mosquito life-cycle and viral incubation period. This predictability is actionable for pre-season vector control.

---

### 💧 4. Humidity is the Strongest Climate Signal
Among climate variables, **humidity** correlates most with case counts. Temperature and precipitation show weaker direct linear correlation — likely because their effect is lagged and non-linear rather than contemporaneous.

---

### ⚰️ 5. Mortality Tracks Caseload Tightly
Deaths correlate with cases at **r ≈ 0.95**. Each epidemic wave carries a proportional mortality cost, underscoring that outbreak prevention — not just treatment capacity — is the key lever for reducing deaths.

---

*Analysis by Mohammed Tahmid Hossain · [saditahmid.github.io](https://saditahmid.github.io) · Related publication: IEEE TENCON 2023*